# 00. Before You Start

© 2026, Anyscale. All Rights Reserved

This notebook gets you from nothing to a working Anyscale workspace with a verified environment: the right versions installed, the right storage understood, and a real GPU worker reached at least once. Notebook 01 assumes all of this already works.

<div class="alert alert-block alert-info">
  <b> Here is the roadmap for this notebook </b>

  <ol>
    <li>What you need before you touch a workspace</li>
    <li>Create a workspace</li>
    <li>Get your code in</li>
    <li>Install dependencies</li>
    <li>Verify the environment: versions, storage, and a GPU worker</li>
    <li>Checklist and next steps</li>
  </ol>
</div>

## 1. What you need before you touch a workspace

- **An Anyscale account, and a cloud already registered to it.** Look for "Clouds" in the left nav of the console. If nothing is there, that's a permissions request to your platform team, not a bug in this notebook.
- **The Anyscale CLI on your laptop**, so you can create and manage workspaces from outside the browser too:

```text
pip install -U anyscale
anyscale login
anyscale cloud list
```

- **Git credentials for a private repo**, if the code you're bringing in lives in one. Section 3 below covers both an HTTPS token and an SSH key.

<div class="alert alert-block alert-info">
<b>On a Kubernetes-backed cloud</b>

Instance type names in a compute config are not cloud VM types there, they're pod sizes your platform team defined when the cloud was set up. Ask them for the compute config name to use and whether your workspace has a GPU cap, before you try to raise worker counts later in this repo. See <code>docs/kubernetes-clouds.md</code> in this repo, and the compute-on-Kubernetes link at the end of this notebook.
</div>

## 2. Create a workspace

A workspace needs three choices. Get these right first, everything else can change later.

| Choice | What to pick first | Why |
|---|---|---|
| Image | `anyscale/image/pytorch-to-ray-train:1` | built from this repo's `containerfile`; torch, torchvision, and tensorboard already pinned, see Section 4 |
| Compute config | `gpu-multinode-dev` | a CPU head node plus GPU worker groups that scale to zero, see Section 5 |
| Idle termination | a short value, e.g. 60 minutes | a workspace left running bills for compute nobody is using |

**In the console:** Workspaces → Create → fill in a name, the image, the compute config, and an idle termination time → Create.

**From the CLI**, the same workspace, described as a file:

```yaml
# Create with: anyscale workspace_v2 create -f jobs/workspace.example.yaml
# Docs: https://docs.anyscale.com/workspaces
name: pytorch-to-ray-train-dev
# Option A: the image built from this repo's containerfile (see notebook 00).
image_uri: anyscale/image/pytorch-to-ray-train:1
# Option B: an Anyscale base image plus requirements installed at start.
# image_uri: anyscale/ray:2.58.0-py312-cu128
# requirements: requirements.txt
compute_config: gpu-multinode-dev   # the registered name on this repo's AWS dev cloud; use as-is there,
                                    # or substitute the equivalent name registered on your own cloud
idle_termination_minutes: 60
```

```text
anyscale workspace_v2 create -f jobs/workspace.example.yaml
```

<div class="alert alert-block alert-warning">
<b>This costs money while it runs.</b> Terminate the workspace when you step away; idle GPU capacity billing does not care whether you're looking at the dashboard. Files in the working directory survive a restart or a termination, they live on persistent storage attached to the workspace, not on any one node.
</div>

## 3. Get your code in

Once the workspace exists, get this repo into it one of two ways.

**Clone it from inside the workspace**, over HTTPS with a personal access token, or over SSH:

```text
# HTTPS, with a token
git clone https://<token>@github.com/<org>/<repo>.git

# SSH: add this public key to your git provider first
cat ~/.ssh/id_rsa.pub
git clone git@github.com:<org>/<repo>.git
```

**Or push it from your laptop**, without cloning inside the workspace at all:

```text
anyscale workspace_v2 push -n <workspace-name> --local-dir .
```

<div class="alert alert-block alert-warning">
The SSH key generated inside a workspace is shared within your organization, not private to you. Treat it accordingly, and prefer an HTTPS token for anything that should stay scoped to you personally.
</div>

## 4. Dependencies

Two ways to get `torch`, `torchvision`, and `tensorboard` into the cluster, at different points on the fast-vs-durable trade-off.

**Fast, for iterating:**

```text
pip install -r requirements.txt
```

Do this once inside the workspace, and also add `requirements.txt` under the Dependencies tab (or `requirements: requirements.txt` in the workspace YAML) so worker nodes and any jobs you submit from this workspace inherit the same packages automatically.

**Durable, for anything you'll run more than once:** bake the packages into an image instead, from this repo's `containerfile`:

```text
# Build with: anyscale image build -n pytorch-to-ray-train -f containerfile --ray-version 2.58.0
# Base images: https://docs.anyscale.com/reference/base-images
FROM anyscale/ray:2.58.0-py312-cu128

# Keep these pins identical to requirements.txt.
RUN pip install --no-cache-dir torch==2.14.0 torchvision==0.29.0 tensorboard==2.21.0
```

```text
anyscale image build -n pytorch-to-ray-train -f containerfile --ray-version 2.58.0
```

Then point `image_uri` at the result, exactly what `jobs/workspace.example.yaml` above already does.

<div class="alert alert-block alert-info">
Runtime environments (<code>pip install</code>, the Dependencies tab, <code>requirements:</code> in a YAML) are for development, where you want to iterate fast. Anyscale Jobs should use a built image: it starts faster, and it's the exact same bytes every time you run it.
</div>

Now confirm the packages this repo needs are actually importable, and see which versions you're on:

In [ ]:
import importlib.metadata as metadata

import mlflow
import ray
import ray.data
import ray.train
import ray.tune
import tensorboard
import torch
import torchvision

for pkg in ["ray", "torch", "torchvision", "mlflow", "tensorboard"]:
    print(f"{pkg:12} {metadata.version(pkg)}")

## 5. Verify the environment

`src/settings.py` centralizes every environment-driven setting this repo's scripts and notebooks read. Import it and ask it to describe itself:

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
print(f"notebook cwd: {Path.cwd()}")
print(f"REPO_ROOT:    {REPO_ROOT}")

from src.settings import Settings, ray_init_with_repo

settings = Settings.from_env()
settings.describe()

In [ ]:
print(f"torch.cuda.is_available() on the head: {torch.cuda.is_available()}")

That prints **False**, and it is supposed to. This compute config's head node (`m5.xlarge`) has no GPU at all, on purpose: the head runs the driver process, the dashboard, and cluster bookkeeping, none of which need an accelerator sitting idle most of the time. GPUs live entirely on worker groups that scale to zero when nothing needs them. Every GPU workload in this repo, starting with the very next notebook, reaches its GPU by asking Ray for one, not by finding one locally.

Storage works the same way: some paths exist on this node only, some are shared across the whole cluster, and some outlive the cluster entirely.

| Path | Scope | Persistence |
|---|---|---|
| `/mnt/local_storage` | this node only | gone when this node terminates |
| `/mnt/cluster_storage` | every node in this cluster | gone when this cluster (workspace or job) terminates |
| `/mnt/user_storage` | every cluster you start in this cloud | persists across restarts and terminations |
| `$ANYSCALE_ARTIFACT_STORAGE` | your cloud's own object store (S3, GCS, or Azure Blob) | persists indefinitely |

In [ ]:
import os

for path in ["/mnt/local_storage", "/mnt/cluster_storage", "/mnt/user_storage"]:
    print(f"{path:26} exists={os.path.isdir(path)}")
print(f"{'$ANYSCALE_ARTIFACT_STORAGE':26} = {os.environ.get('ANYSCALE_ARTIFACT_STORAGE')}")
print(f"{'settings.storage_path':26} = {settings.storage_path}")

Now reach an actual GPU. `ray_init_with_repo()` does what a bare `ray.init()` would, plus one thing a workspace notebook specifically needs: it ships this repo to every worker as the cluster's `working_dir`, so `from src.xxx import yyy` works on a worker process too, not just in this notebook. See its docstring in `src/settings.py` for why a bare `ray.init()` fails here in a way it wouldn't inside an Anyscale Job.

In [ ]:
ray_init_with_repo()

In [ ]:
import time

import ray


@ray.remote(num_gpus=1)
def gpu_probe():
    import socket

    return socket.gethostname(), torch.cuda.get_device_name(0)


start = time.time()
results = ray.get([gpu_probe.remote(), gpu_probe.remote()], timeout=900)
elapsed = time.time() - start

hosts = {host for host, _ in results}
print(f"elapsed: {elapsed:.1f}s")
for host, gpu in results:
    print(f"  {host}: {gpu}")
print(f"distinct nodes used: {len(hosts)}")

<div class="alert alert-block alert-info">
If that cell took a while, that's the autoscaler working, not a stall. This compute config's GPU worker groups start at zero nodes; the first request for a GPU makes Anyscale provision one, which takes real minutes, not seconds. Run the cell again right after and it comes back fast, because the node is already warm.
</div>

In [ ]:
for node in ray.nodes():
    if not node["Alive"]:
        continue
    resources = node.get("Resources", {})
    print(f"{node['NodeManagerHostname']:35} CPU={resources.get('CPU', 0):.0f}  GPU={resources.get('GPU', 0):.0f}")

## 6. Checklist and next steps

Before moving on to notebook 01, all of the following should be true:

- [ ] You can create, reach, and terminate an Anyscale workspace.
- [ ] Your code is inside it, either cloned with git or pushed from your laptop.
- [ ] `torch`, `torchvision`, `ray.train`, `ray.tune`, `ray.data`, `mlflow`, and `tensorboard` all import, either via `pip install -r requirements.txt` or a built image.
- [ ] You understand why `torch.cuda.is_available()` is `False` on the head, and you know where `settings.storage_path` points.
- [ ] You reached a GPU worker with `ray.get(...)` and saw two distinct hostnames above.

Continue to **`01_pytorch_as_is.ipynb`**, where the same head node runs a plain PyTorch training loop by placing it, unchanged, on a GPU worker.

## Further reading

| Resource | Why |
|---|---|
| [Get started](https://docs.anyscale.com/get-started) | the Anyscale onboarding overview |
| [Create a workspace](https://docs.anyscale.com/get-started/create-workspace) | step by step workspace creation |
| [Workspaces](https://docs.anyscale.com/workspaces) | the full workspace reference |
| [Git in workspaces](https://docs.anyscale.com/workspaces/git) | cloning private repos, SSH keys |
| [Notebooks in workspaces](https://docs.anyscale.com/workspaces/notebooks) | Jupyter inside a workspace |
| [VS Code](https://docs.anyscale.com/workspaces/vscode) | the other workspace editor |
| [Dependency management](https://docs.anyscale.com/dependency-management) | requirements vs. images, the trade-off in Section 4 |
| [Runtime environments](https://docs.anyscale.com/dependency-management/runtime-environment) | the fast, development-time path |
| [Build an image](https://docs.anyscale.com/container-image/build-image) | the durable path, `anyscale image build` |
| [Bring your own image](https://docs.anyscale.com/container-image/custom-image) | using an image built outside this flow |
| [Base images](https://docs.anyscale.com/reference/base-images) | what `FROM anyscale/ray:...` gives you |
| [Compute configs](https://docs.anyscale.com/configuration/compute) | head and worker shapes, like `gpu-multinode-dev` |
| [Compute configs on Kubernetes](https://docs.anyscale.com/configuration/compute/kubernetes) | pod sizes instead of VM types |
| [Storage](https://docs.anyscale.com/storage) | the full picture behind the table in Section 5 |
| [CLI reference](https://docs.anyscale.com/reference/cli) | every `anyscale` command used above |
| [Template: workspace-intro](https://github.com/anyscale/templates/tree/main/templates/workspace-intro) | a guided first workspace, in more depth than this notebook |
| [Template: workspaces-dev-flow](https://github.com/anyscale/templates/tree/main/templates/workspaces-dev-flow) | day-to-day habits once the workspace exists |
| [Template: compute-config-cluster-shapes](https://github.com/anyscale/templates/tree/main/templates/compute-config-cluster-shapes) | choosing head and worker shapes for other workloads |
| [Template: storage-access-large-datasets](https://github.com/anyscale/templates/tree/main/templates/storage-access-large-datasets) | reading data too large to fit on one node |
| [Ray tasks](https://docs.ray.io/en/latest/ray-core/tasks.html) | the `@ray.remote` primitive the GPU probe used above |
| [Ray dashboard](https://docs.ray.io/en/latest/ray-observability/getting-started.html) | watching the autoscaler add the nodes you just waited on |